In [1]:
!pip install pandas requests numpy --quiet

import pandas as pd
import numpy as np
import io
import os
import requests
from datetime import datetime
from google.colab import files
import time

print("✅ Librairies chargées.")

# =============================================================================
# CHARGEMENT DU FICHIER stations_summary.csv
# =============================================================================

print("\n📂 Téléchargez votre fichier stations_summary.csv :")
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError("❌ Aucun fichier téléchargé.")

filename = list(uploaded.keys())[0]
df_stations = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))

COLS_REQUISES = {'nom', 'lat', 'lon', 'depuis'}
cols_csv = set(df_stations.columns.str.strip().str.lower())
manquantes = COLS_REQUISES - cols_csv
if manquantes:
    raise ValueError(f"❌ Colonnes manquantes : {manquantes}")

df_stations.columns = df_stations.columns.str.strip().str.lower()

REGIONS = {}
for i, row in df_stations.iterrows():
    rid = f'R{i+1:02d}'
    REGIONS[rid] = {
        'nom': str(row['nom']).strip(),
        'lat': float(row['lat']),
        'lon': float(row['lon']),
        'depuis': int(row['depuis']),
    }

print(f"\n✅ {len(REGIONS)} station(s) chargée(s)")

# =============================================================================
# PARAMÈTRES & CONSTANTES
# =============================================================================

ANNEE_FIN = 2024
TIMEOUT = 60
MAX_RETRIES = 3
DELAI_DEFAUT = 2  # Délai par défaut entre requêtes (secondes)
DELAI_RATE_LIMIT = 5  # Délai si rate limiting détecté

API_OPEN_METEO = "https://archive-api.open-meteo.com/v1/archive"

print(f"⚙️  Extraction avec gestion du rate limiting")

# =============================================================================
# SESSION REQUESTS AVEC RETRY
# =============================================================================

session = requests.Session()

# =============================================================================
# FONCTION D'EXTRACTION OPEN-METEO (OPTIMISÉE)
# =============================================================================

def extraire_open_meteo(region_id, lat, lon, annee_debut, annee_fin=ANNEE_FIN):
    """
    Extrait données HORAIRES d'Open-Meteo avec gestion du rate limiting.
    """
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': f'{annee_debut}-01-01',
        'end_date': f'{annee_fin}-12-31',
        'hourly': ','.join([
            'temperature_2m',
            'precipitation',
            'relative_humidity_2m',
            'wind_speed_10m',
            'shortwave_radiation',
        ]),
        'timezone': 'UTC'
    }

    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(
                API_OPEN_METEO,
                params=params,
                timeout=TIMEOUT
            )

            # Si rate limit détecté
            if response.status_code == 429:
                wait_time = DELAI_RATE_LIMIT * (attempt + 1)
                print(f"\n   ⏳ Rate limit détecté — Attente {wait_time}s...", end='', flush=True)
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            data = response.json()

            if 'hourly' not in data:
                print(f"\n   ⚠️  {region_id} : Réponse invalide d'Open-Meteo (pas de données horaires)")
                return pd.DataFrame()

            hourly = data['hourly']
            times = pd.to_datetime(hourly['time'])

            df_hourly = pd.DataFrame({
                'datetime': times,
                'region_id': region_id,
                'om_temp_C_hourly': hourly.get('temperature_2m', [None] * len(times)),
                'om_precip_mm_hourly': hourly.get('precipitation', [None] * len(times)),
                'om_humidity_pct_hourly': hourly.get('relative_humidity_2m', [None] * len(times)),
                'om_wind_speed_ms_hourly': hourly.get('wind_speed_10m', [None] * len(times)),
                'om_radiation_MJm2_hourly': np.array(hourly.get('shortwave_radiation', [None] * len(times))) / 1000 if 'shortwave_radiation' in hourly else [None] * len(times),
            })

            return df_hourly

        except requests.exceptions.Timeout:
            print(f"\n   ⏱️  Timeout (tentative {attempt+1}/{MAX_RETRIES})", end='', flush=True)
            time.sleep(DELAI_RATE_LIMIT)
        except requests.exceptions.ConnectionError as e:
            print(f"\n   🔌 Erreur réseau (tentative {attempt+1}/{MAX_RETRIES})", end='', flush=True)
            time.sleep(DELAI_RATE_LIMIT)
        except Exception as e:
            print(f"\n   ❌ Erreur Open-Meteo : {str(e)[:60]}")
            return pd.DataFrame()

    return pd.DataFrame()

# =============================================================================
# FONCTION D'EXTRACTION GIOVANNI NASA (ALTERNATIVE à POWER)
# =============================================================================

def extraire_nasa_giovanni(region_id, lat, lon, annee_debut, annee_fin=ANNEE_FIN):
    """
    Extrait données de MERRA-2 via Giovanni API (alternative à POWER).
    Variables : température, précipitations, rayonnement.
    """
    # MERRA-2 Monthly Mean données
    base_url = "https://giovanni.gsfc.nasa.gov/daac-bin/G4.pl"

    params = {
        'service': 'SUBSET_MERRA2',
        'portal': 'Giovanni',
        'variableFacets': 'T2M,PRECTOT,SWGDN',
        'location': f'POINT({lon},{lat})',
        'startTime': f'{annee_debut}-01-01T00:00:00Z',
        'endTime': f'{annee_fin}-12-31T23:59:59Z',
        'result': 'json',
    }

    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        print(f"\n   ℹ️  Giovanni API non directement accessible (optionnel)")
        return pd.DataFrame()
    except:
        return pd.DataFrame()

# =============================================================================
# FONCTION ALTERNATIVE : DONNÉES INTERPOLÉES À PARTIR D'OM
# =============================================================================

def completer_donnees_ommeteo(df_om):
    """
    Génère des variables manquantes à partir d'Open-Meteo.
    """
    if df_om.empty:
        return df_om

    # Estimation de la pression (formule barométrique approx.)
    if 'altitude' in df_om.columns:
        alt = df_om.get('altitude', 0)
        df_om['np_pressure_hPa'] = 1013.25 * (1 - 0.0065 * alt / 288.15) ** 5.255

    # Estimation du déficit de saturation
    df_om['np_vapor_pressure'] = (
        df_om['om_humidity_pct_hourly'] / 100 * # Changed to hourly column
        6.112 * np.exp(17.62 * df_om['om_temp_C_hourly'] / (243.12 + df_om['om_temp_C_hourly'])) # Changed to hourly column
    )

    return df_om

# =============================================================================
# LANCEMENT DE L'EXTRACTION
# =============================================================================

dfs_resultats = []

for i, (rid, info) in enumerate(REGIONS.items()):
    print(f"\n🌍 [{i+1}/{len(REGIONS)}] {rid} : {info['nom']:25s}", end='', flush=True)

    # Open-Meteo
    df_om = extraire_open_meteo(
        region_id=rid,
        lat=info['lat'],
        lon=info['lon'],
        annee_debut=info['depuis'],
    )

    if len(df_om) > 0:
        df_om['region_nom'] = info['nom']
        df_om['lat'] = info['lat']
        df_om['lon'] = info['lon']

        # Compléter avec variables calculées
        df_om = completer_donnees_ommeteo(df_om)

        dfs_resultats.append(df_om)
        print(f" ✅ {len(df_om)} heures") # Changed 'mois' to 'heures'
    else:
        print(f" ❌ Échec")

    # Délai adaptatif pour éviter rate limiting
    if (i + 1) % 5 == 0:
        print(f"   ⏳ Pause {DELAI_RATE_LIMIT}s...")
        time.sleep(DELAI_RATE_LIMIT)
    else:
        time.sleep(DELAI_DEFAUT)

# =============================================================================
# FUSION & FINITION
# =============================================================================

if dfs_resultats:
    df_final = pd.concat(dfs_resultats, ignore_index=True)
    df_final = df_final.sort_values(['region_nom', 'datetime']).reset_index(drop=True)

    print(f"\n✅ Extraction terminée")
    print(f"   📊 {len(df_final)} lignes × {df_final.shape[1]} colonnes")
    print(f"   🏠 {df_final['region_nom'].nunique()} stations")
    print(f"   📅 {df_final['datetime'].min().date()} → {df_final['datetime'].max().date()}")

else:
    raise Exception("❌ Aucune donnée extraite — Vérifiez votre connexion")

# =============================================================================
# SAUVEGARDE
# =============================================================================

OUTPUT_CSV = f'extraction_open_meteo_hourly_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv' # Changed filename
df_final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f"\n💾 {OUTPUT_CSV}")
files.download(OUTPUT_CSV)

# =============================================================================
# APERÇU RAPIDE
# =============================================================================
# Removed the summary section as it was for monthly data and would need significant changes for hourly.

✅ Librairies chargées.

📂 Téléchargez votre fichier stations_summary.csv :


Saving stations_summary.csv to stations_summary.csv

✅ 24 station(s) chargée(s)
⚙️  Extraction avec gestion du rate limiting

🌍 [1/24] R01 : ksar El Kebir             ✅ 113976 heures

🌍 [2/24] R02 : DAR ELGUEDDARI            ✅ 113976 heures

🌍 [3/24] R03 : SIDI ALLAL TAZI           ✅ 87672 heures

🌍 [4/24] R04 : SUTA - FERME ABT          ✅ 87672 heures

🌍 [5/24] R05 : SUTA-TAZEROUALT          
   ⏳ Rate limit détecté — Attente 5s...
   ⏳ Rate limit détecté — Attente 10s...
   ⏳ Rate limit détecté — Attente 15s... ❌ Échec
   ⏳ Pause 5s...

🌍 [6/24] R06 : SUTA OULAD AYAD           ✅ 87672 heures

🌍 [7/24] R07 : MECHRAA BELKSIRI          ✅ 61368 heures

🌍 [8/24] R08 : SUTA 507                  ✅ 52608 heures

🌍 [9/24] R09 : M-Zaio                    ✅ 78912 heures

🌍 [10/24] R10 : Merksen                   ✅ 70128 heures
   ⏳ Pause 5s...

🌍 [11/24] R11 : SUTA_CENTAGRI             ✅ 70128 heures

🌍 [12/24] R12 : SUTA INRA                
   ⏳ Rate limit détecté — Attente 5s...
   ⏳ Rate li

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>